# 🚀 Retrain — DEBUG MODE
All-in-one cell. Run ONE cell only.

In [ ]:
!pip install -q ultralytics kagglehub pyyaml
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}' if torch.cuda.is_available() else 'NO GPU!')

In [ ]:
import kagglehub, os, shutil, yaml, xml.etree.ElementTree as ET, time
from pathlib import Path
from ultralytics import YOLO

###############################################
# STEP 1: DOWNLOAD
###############################################
print('='*60)
print('STEP 1: DOWNLOAD DATASET')
print('='*60)

src = Path(kagglehub.dataset_download('habibiahmadim09/kerusakan-jalan'))
print(f'Downloaded to: {src}')
print(f'Exists: {src.exists()}')

# Show EVERYTHING in source
print(f'\nContents of {src}:')
all_files = list(src.rglob('*'))
print(f'Total items: {len(all_files)}')

# Show by extension
from collections import Counter
exts = Counter()
for f in all_files:
    if f.is_file():
        exts[f.suffix.lower()] += 1
print(f'File types: {dict(exts)}')

# Show first 30 files
print(f'\nFirst 30 files:')
for f in sorted(all_files)[:30]:
    rel = f.relative_to(src)
    print(f'  {rel}')

# Show structure per split
for split in ['train', 'valid', 'test']:
    d = src / split
    if d.exists():
        files = list(d.rglob('*'))
        imgs = [f for f in files if f.suffix.lower() in ['.jpg','.jpeg','.png','.bmp']]
        xmls = [f for f in files if f.suffix == '.xml']
        print(f'\n{split}/: {len(files)} total, {len(imgs)} images, {len(xmls)} xmls')
        if imgs:
            print(f'  Sample image: {imgs[0].relative_to(src)}')
        if xmls:
            print(f'  Sample xml: {xmls[0].relative_to(src)}')

###############################################
# STEP 2: CLONE REPO
###############################################
print(f'\n{"="*60}')
print('STEP 2: CLONE REPO')
print('='*60)

if not Path('jalancerdas-ai').exists():
    !git clone -q https://github.com/Srjeff27/jalancerdas-ai.git
    print('Cloned!')
else:
    print('Already exists')

###############################################
# STEP 3: PREPARE DATASET
###############################################
print(f'\n{"="*60}')
print('STEP 3: PREPARE DATASET')
print('='*60)

CLASS_MAP = {
    'retak_memanjang': 0,
    'pengelupasan_lapisan_permukaan': 1,
    'lubang': 2,
    'retak_kulit_buaya': 3,
    'retak_blok': 4,
    'retak_pinggir': 5,
}

ds = Path('jalancerdas-ai/ai-model/dataset')
if ds.exists():
    shutil.rmtree(ds)
    print('Cleaned old dataset')

total_imgs = 0
total_lbls = 0

for src_s, dst_s in [('train','train'),('valid','val'),('test','test')]:
    sd = src / src_s
    if not sd.exists():
        print(f'{src_s}/ NOT FOUND - skipping')
        continue
    
    di = ds/dst_s/'images'
    dl = ds/dst_s/'labels'
    di.mkdir(parents=True, exist_ok=True)
    dl.mkdir(parents=True, exist_ok=True)
    
    # Find images
    imgs = []
    for ext in ['*.jpg','*.jpeg','*.png','*.bmp']:
        imgs.extend(sd.rglob(ext))
    
    xml_map = {x.stem: x for x in sd.rglob('*.xml')}
    
    ni, nl = 0, 0
    for img in imgs:
        try:
            shutil.copy2(img, di/img.name)
            ni += 1
        except Exception as e:
            print(f'  COPY ERROR: {img.name}: {e}')
        
        xml = xml_map.get(img.stem)
        if xml:
            try:
                t = ET.parse(xml)
                r = t.getroot()
                w = int(r.find('size/width').text)
                h = int(r.find('size/height').text)
                ls = []
                for o in r.findall('object'):
                    nm = o.find('name').text
                    if nm not in CLASS_MAP: continue
                    b = o.find('bndbox')
                    x1,y1 = float(b.find('xmin').text), float(b.find('ymin').text)
                    x2,y2 = float(b.find('xmax').text), float(b.find('ymax').text)
                    ls.append(f'{CLASS_MAP[nm]} {((x1+x2)/2)/w:.6f} {((y1+y2)/2)/h:.6f} {(x2-x1)/w:.6f} {(y2-y1)/h:.6f}')
                (dl/(img.stem+'.txt')).write_text('\n'.join(ls))
                nl += len(ls)
            except Exception as e:
                print(f'  XML ERROR: {xml.name}: {e}')
        else:
            (dl/(img.stem+'.txt')).write_text('')
    
    total_imgs += ni
    total_lbls += nl
    print(f'{src_s} -> {dst_s}: {ni} images, {nl} labels')

print(f'\nTotal: {total_imgs} images, {total_lbls} labels')

###############################################
# STEP 4: VERIFY
###############################################
print(f'\n{"="*60}')
print('STEP 4: VERIFY')
print('='*60)

for s in ['train','val','test']:
    p = ds/s/'images'
    if p.exists():
        files = list(p.glob('*'))
        n = len(files)
        print(f'{s}/images: {n} files {"✅" if n > 0 else "❌"}')
        if n > 0:
            print(f'  Sample: {files[0].name}')
    else:
        print(f'{s}/images: ❌ DIR NOT FOUND')

###############################################
# STEP 5: FIX data.yaml
###############################################
yp = Path('jalancerdas-ai/ai-model/configs/data.yaml')
dp = str(Path.cwd()/ds)
with open(yp) as f: d = yaml.safe_load(f)
d['path'] = dp
with open(yp,'w') as f: yaml.dump(d, f, default_flow_style=False)
print(f'\ndata.yaml: {dp}')

DATA_YAML = str(yp)

if total_imgs == 0:
    print('\n❌ NO IMAGES FOUND! Cannot train.')
    print('Check the source structure above.')
else:
   ###############################################
    # STEP 6: TRAIN
    ###############################################
    print(f'\n{"="*60}')
    print('STEP 6: TRAIN YOLOv8s')
    print('='*60)
    print('30-50 minutes... DO NOT CLOSE TAB!\n')
    
    start = time.time()
    model = YOLO('yolov8s.pt')
    results = model.train(
        data=DATA_YAML, epochs=200, batch=16, imgsz=640, device=0,
        patience=50, optimizer='SGD', lr0=0.01, lrf=0.01,
        mosaic=1.0, mixup=0.15, degrees=10.0, translate=0.1,
        scale=0.5, fliplr=0.5, copy_paste=0.1, close_mosaic=15,
        project='jalancerdas-ai/ai-model/runs', name='train_v2', exist_ok=True
    )
    
    elapsed = int(time.time() - start)
    BEST = 'jalancerdas-ai/ai-model/runs/train_v2/weights/best.pt'
    print(f'\nDONE in {elapsed//60}m {elapsed%60}s — {BEST}')
    
   ###############################################
    # STEP 7: EVALUATE
    ###############################################
    print(f'\n{"="*60}')
    print('STEP 7: EVALUATE')
    print('='*60)
    
    model = YOLO(BEST)
    r = model.val(data=DATA_YAML, device=0)
    print(f'\nmAP50={r.box.map50:.4f} Prec={r.box.mp:.4f} Recall={r.box.mr:.4f}')
    
   ###############################################
    # STEP 8: EXPORT
    ###############################################
    print(f'\n{"="*60}')
    print('STEP 8: EXPORT TFLite')
    print('='*60)
    
    ep = model.export(format='tflite', imgsz=640, half=True, simplify=True)
    print(f'TFLite: {ep} ({os.path.getsize(ep)/1024/1024:.1f} MB)')
    
   ###############################################
    # STEP 9: DOWNLOAD
    ###############################################
    print(f'\n{"="*60}')
    print('STEP 9: DOWNLOAD')
    print('='*60)
    
    from google.colab import files
    files.download(ep)
    files.download(BEST)
    print('\n🎉 ALL DONE!')
    print('Copy best.tflite -> mobile/assets/models/pothole_yolo.tflite')